In [ ]:
# --- snatac-cre-benchmark repo config ---
# Override with:  export PROJECT_ROOT=/path/to/project ; export DATA_ROOT=$PROJECT_ROOT/output_so
import os
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", ".")
DATA_ROOT    = os.environ.get("DATA_ROOT",    os.path.join(PROJECT_ROOT, "output_so"))


# Check Cell ranger's output files

In [ ]:
!ls $DATA_ROOT/cell_ranger_output/sample1/outs/


# QC samples and cells

In [ ]:
import snapatac2 as snap

snap.__version__

## Check samples

### Distribution of fragment sizes (Sample Check)

In [ ]:
import os
import logging
import matplotlib.pyplot as plt
import snapatac2 as snap
from anndata import AnnData
from typing import Optional  # Python 3.9 이하 호환

def frag_size_distr_matplotlib(
    adata: AnnData,                      # 인자명을 adata로 일관
    use_rep: str = "frag_size_distr",
    max_recorded_size: int = 1000,
    ax: Optional[plt.Axes] = None,       # Optional 사용
) -> None:
    """Plot fragment size distribution using matplotlib."""
    # 이미 계산된 분포가 없거나 길이가 부족하면 새로 계산
    if use_rep not in adata.uns or len(adata.uns[use_rep]) < max_recorded_size:
        logging.info("Computing fragment size distribution...")
        snap.metrics.frag_size_distr(
            adata, add_key=use_rep, max_recorded_size=max_recorded_size
        )

    dist = adata.uns[use_rep]            # 덮어쓰기 피하려고 이름 변경
    x, y = zip(*enumerate(dist))

    # 전달받은 축이 없으면 새 축 생성
    if ax is None:
        _, ax = plt.subplots()

    # 첫 번째 bin(인덱스 0)은 제외하여 원래 동작과 동일하게
    ax.plot(x[1:], y[1:])
    ax.set_xlabel("Fragment size")
    ax.set_ylabel("Count")

# sample1~sample9의 fragment 파일 경로 목록
sample_files = [
    f"{DATA_ROOT}/cell_ranger_output/sample{i}/outs/fragments.tsv.gz"
    for i in range(1, 10)
]

# 3x3 서브플롯 격자 생성
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i, fragment_file in enumerate(sample_files, start=1):

    # 데이터 가져오기 (디스크에도 저장)
    adata = snap.pp.import_data(
        fragment_file,
        chrom_sizes=snap.genome.hg38,
        file=None,
        sorted_by_barcode=False,
    )

    # 해당 위치의 서브플롯 선택
    ax = axes[(i - 1) // 3, (i - 1) % 3]
    frag_size_distr_matplotlib(adata, ax=ax)
    ax.set_title(f"Sample {i}")

# 레이아웃 정리 및 표시
plt.tight_layout()
plt.show()

In [ ]:
import os
import snapatac2 as snap
import matplotlib.pyplot as plt
import numpy as np

# sample1~sample9의 fragments 경로
fragment_files = [
    f"{DATA_ROOT}/cell_ranger_output/sample{i}/outs/fragments.tsv.gz"
    for i in range(1, 10)
]

all_tsse_scores = []

for i, fragment_file in enumerate(fragment_files, start=1):
    adata = snap.pp.import_data(
        fragment_file,
        chrom_sizes=snap.genome.hg38,
        sorted_by_barcode=False,
    )

    # TSS 점수 계산
    snap.metrics.tsse(adata, snap.genome.hg38)
    all_tsse_scores.append(list(adata.obs["tsse"]))

# 플롯
x_values = np.arange(len(all_tsse_scores))
plt.figure(figsize=(12, 8))
plt.violinplot(all_tsse_scores, positions=x_values, showmeans=False, showmedians=True)

# 기준선 (TSS enrichment = 6)
plt.axhline(y=6, color="red", linestyle="--", label="TSS Enrichment Score = 6")

plt.xlabel("Sample Index")
plt.ylabel("TSS Enrichment Score")
plt.title("Violin Plot of TSS Scores Across 9 Samples")
plt.xticks(ticks=x_values, labels=[f"Sample {i}" for i in range(1, 10)])
plt.legend()
plt.show()

### Unique fragments (Cell Check)

In [ ]:
#Sample 각각 실행해야함. 
fragment_file = f'{DATA_ROOT}/cell_ranger_output/sample{i}/outs/fragments.tsv.gz'
data = snap.pp.import_data(
    fragment_file,
    chrom_sizes=snap.genome.hg38,
    sorted_by_barcode=False,
)

In [ ]:
snap.metrics.tsse(data, snap.genome.hg38)

In [ ]:
snap.pl.tsse(data, interactive=False)
#snap.pl.tsse는 축(ax)을 받지 않아서 3×3 한 장에 직접 배치할 수 없음.

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import snapatac2 as snap
from matplotlib.figure import Figure

# 1) 경로 설정
base = f"{DATA_ROOT}/cell_ranger_output"
fragment_files = [f"{base}/sample{i}/outs/fragments.tsv.gz" for i in range(1, 10)]
png_dir = f"{DATA_ROOT}/tsse_png"
os.makedirs(png_dir, exist_ok=True)

saved_pngs = []

# 2) 샘플별로 snap.pl.tsse 실행 → PNG 저장
for i, frag in enumerate(fragment_files, start=1):
    if not os.path.exists(frag):
        print(f"[WARN] Missing: {frag}")
        saved_pngs.append(None)
        continue

    # import_data 경고를 피하려면 import_fragments 권장
    adata = snap.pp.import_fragments(
        frag,
        chrom_sizes=snap.genome.hg38,
        sorted_by_barcode=False,
    )
    snap.metrics.tsse(adata, snap.genome.hg38)

    # ▶ 핵심: 반환된 Figure를 직접 받아서 저장
    plt.close('all')  # 이전 그림 정리
    ret = snap.pl.tsse(adata, interactive=False)     # 보통 Figure를 반환
    fig = ret if isinstance(ret, Figure) else plt.gcf()

    out_png = os.path.join(png_dir, f"sample{i}_tsse.png")
    fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    saved_pngs.append(out_png)

# 3) 저장된 이미지들을 3×3 모자이크로 표시
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for i, ax in enumerate(axes.flat, start=1):
    p = saved_pngs[i-1] if i-1 < len(saved_pngs) else None
    if p and os.path.exists(p):
        ax.imshow(mpimg.imread(p))
        ax.set_title(f"Sample {i}")
    else:
        ax.text(0.5, 0.5, f"Sample {i}\n(missing)", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Filtering Cell counts

In [ ]:
import os
import snapatac2 as snap
import pandas as pd

tsse_dic_list = {}
qc_df_dic = {}

h5_dir = f"{DATA_ROOT}/h5ad"
os.makedirs(h5_dir, exist_ok=True)

for i in range(1, 10):
    frag = f"{DATA_ROOT}/cell_ranger_output/sample{i}/outs/fragments.tsv.gz"
    out  = f"{h5_dir}/sample{i}.h5ad"

    # (선택) 기존 h5ad 삭제
    if os.path.exists(out):
        os.remove(out)

    # fragments 로드 + TSSE 계산
    adata = snap.pp.import_fragments(
        frag,
        chrom_sizes=snap.genome.hg38,
        sorted_by_barcode=False,
    )
    snap.metrics.tsse(adata, snap.genome.hg38)

    # h5ad로 저장(캐시)
    adata.write_h5ad(out)

    # barcode → TSSE 매핑 저장
    tsse_dic_list[f"sample{i}"] = dict(zip(adata.obs_names, adata.obs["tsse"]))

    # QC 로드 후 TSSE 붙여서 딕셔너리에 보관
    qc_file = f"{DATA_ROOT}/cell_ranger_output/sample{i}/outs/singlecell.csv"
    qc_df = pd.read_csv(qc_file, index_col="barcode")
    # 필요 시 바코드 포맷 맞추기: qc_df.index = qc_df.index.str.replace("-1$", "", regex=True)
    qc_df["TSS_enrichment"] = qc_df.index.map(tsse_dic_list[f"sample{i}"])
    qc_df_dic[f"sample{i}"] = qc_df

print("✅ Done. h5ad saved to:", h5_dir)

In [ ]:
import pandas as pd

# 결과를 저장할 딕셔너리 초기화
filtered_cell_counts = {}

umi_min = 5000
umi_max = 50000

for i, sample in enumerate(sorted(qc_df_dic.keys())):
    qc_df = qc_df_dic[sample]
    
    filtered_cells = qc_df[
        (qc_df['TSS_enrichment'] >= 6) & 
        (qc_df['passed_filters'] + 1 >= umi_min) & 
        (qc_df['passed_filters'] + 1 <= umi_max)
    ]
    
    filtered_cell_counts[sample] = len(filtered_cells)

for sample, count in filtered_cell_counts.items():
    print(f"{sample}: {count} cells")


In [ ]:
import pandas as pd

umi_min = 5000
umi_max = 50000

pre_counts = {}       # 필터 전
post_counts = {}      # 필터 후

for sample, qc_df in qc_df_dic.items():
    # 필터 전
    pre_counts[sample] = len(qc_df)  # 또는 qc_df.shape[0]

    # 필터 후 
    filtered = qc_df[
        (qc_df['TSS_enrichment'] >= 6) &
        (qc_df['passed_filters'] + 1 >= umi_min) &
        (qc_df['passed_filters'] + 1 <= umi_max)
    ]
    post_counts[sample] = len(filtered)

summary = (
    pd.DataFrame({"pre_cells": pre_counts, "post_cells": post_counts})
      .assign(retention=lambda d: d["post_cells"] / d["pre_cells"])
      .sort_index()
)

print(summary)
print("\nTOTAL:",
      "pre =", summary["pre_cells"].sum(),
      "post =", summary["post_cells"].sum(),
      "retention =", (summary["post_cells"].sum() / summary["pre_cells"].sum()))

In [ ]:
import pandas as pd
import os

# 결과를 저장할 딕셔너리 초기화
filtered_cell_counts = {}

umi_min = 5000
umi_max = 50000

# 결과를 저장할 디렉터리 설정
output_dir = f"{DATA_ROOT}/filtered_samples"
os.makedirs(output_dir, exist_ok=True)

for i, sample in enumerate(sorted(qc_df_dic.keys())):
    qc_df = qc_df_dic[sample]
    
    # 'is__cell_barcode == 1'인 고품질 셀만 필터링
    qc_df_cells = qc_df.query('is__cell_barcode == 1')
    
    # TSS enrichment score와 UMI 조건을 만족하는 셀 필터링
    filtered_cells = qc_df_cells[
        (qc_df_cells['TSS_enrichment'] >= 6) & 
        (qc_df_cells['passed_filters'] + 1 >= umi_min) & 
        (qc_df_cells['passed_filters'] + 1 <= umi_max)
    ]
    
    # 필터링된 데이터를 파일로 저장
    output_file = os.path.join(output_dir, f"{sample}_filtered.csv")
    filtered_cells.to_csv(output_file, index=True)
    print(f"Filtered data for {sample} saved to {output_file}")


### Mitochondrial fraction

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def draw_qc(y_label="TSS_fragments", y_lim=(0, 0.6), y_coff=0.2, nrows=3, ncols=3, size=5):
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * size, nrows * size))

    for i, sample in enumerate(sorted(qc_df_dic.keys())):
        
        plt.sca(axes[int(i/ncols)][i%ncols])

        qc_df = qc_df_dic[sample]
        qc_df_cells = qc_df.query('is__cell_barcode == 1')
        
        frag   = qc_df[y_label] + 1
        umi    = qc_df['passed_filters'] + 1
        frac   = frag / umi
        logumi = np.log10(umi + 1)
        
        # Visualize 전체 셀 데이터
        plt.scatter(logumi, frac, alpha=0.4, s=1, label="All Cells")
        
        frag   = qc_df_cells[y_label] + 1
        umi    = qc_df_cells['passed_filters'] + 1
        frac   = frag / umi
        logumi = np.log10(umi + 1)
        
        # Visualize 필터링된 셀 데이터
        plt.scatter(logumi, frac, s=1, label="Filtered Cells")
        
        plt.xlim(2, 5)
        plt.ylim(y_lim)
        plt.title(sample)
        plt.xlabel("log10(UMI+1)")
        plt.ylabel(f"{y_label}/{y_label} + UMI")
        plt.axhline(y=y_coff, color='r', linestyle='-', lw=0.5)
        plt.axvline(x=3, color='r', linestyle='-', lw=0.5)

    fig.text(0.5, 0, 'log10(UMI+1)', ha='center')
    fig.text(0, 0.5, f'Fraction of {y_label}', va='center', rotation='vertical')
    plt.tight_layout()
    plt.show()

# Mitochondrial 비율 시각화
draw_qc("mitochondrial", y_lim=(0, 1), y_coff=0.1)
